In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets
!pip install optuna optuna-integration wandb

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ahq9ehmp/unsloth_4741afec2565419aafd1171196630065
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ahq9ehmp/unsloth_4741afec2565419aafd1171196630065
  Resolved https://github.com/unslothai/unsloth.git to commit 5c473fab80e079bb525345b86cb71afd409262c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 157.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 24.4 MB/s eta 0:00:0

In [ ]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch, wandb, os, gc
from google.colab import userdata, drive

In [ ]:
# 1. System Prep
drive.mount('/content/drive')
wandb.login(key=userdata.get('WANDB_API_KEY'))
os.environ["WANDB_PROJECT"] = "medical-llama-3.1-8b-instruct-lora-bf16"
os.environ["WANDB_LOG_MODEL"] = "true" # Saves bandwidth/compute

Mounted at /content/drive


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: akhilgalla (foobar41) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# 2. Model Loading (16-bit Native)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Meta-Llama-3.1-8B-Instruct",
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16,
    load_in_4bit=False,
)

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct as a legacy tokenizer.


In [ ]:
# # 1. Disable automatic BOS/EOS insertion since SFTTrainer adds the special tokens. Prevent double BOS/EOS
# tokenizer.add_bos_token = False
# tokenizer.add_eos_token = False

# # 2. Standard Llama 3 padding configuration
# tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# 3. Winning LoRA Config (Trial 6 Results)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407
)

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
def format_chat(example):
  prompt_messages = [
      {"role": "system", "content": "You are a highly accurate medical AI assistant."},
      {"role": "user", "content": example["input"]}
  ]
  prompt_str = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
  completion_str = example["output"] + "<|eot_id|>"

  return {"prompt": prompt_str, "completion": completion_str}

In [ ]:
# 4. Dataset Prep (34,000 Medical Flashcards)
dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards", split="train")

full_dataset = dataset.map(format_chat, remove_columns=dataset.column_names)

In [ ]:
# tokenize=False, add_generation_prompt=False
full_dataset[0]

{'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a highly accurate medical AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?<|eot_id|>',
 'completion': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.<|eot_id|>'}

In [ ]:
# tokenize=False, add_generation_prompt=True
full_dataset[0]

{'prompt': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nYou are a highly accurate medical AI assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is the relationship between very low Mg2+ levels, PTH levels, and Ca2+ levels?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n',
 'completion': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.<|eot_id|>'}

In [ ]:
# tokenize=True, add_generation_prompt=True
full_dataset[0]

{'prompt': [128000,
  128006,
  9125,
  128007,
  271,
  38766,
  1303,
  33025,
  2696,
  25,
  6790,
  220,
  2366,
  18,
  198,
  15724,
  2696,
  25,
  220,
  1627,
  10263,
  220,
  2366,
  19,
  271,
  2675,
  527,
  264,
  7701,
  13687,
  6593,
  15592,
  18328,
  13,
  128009,
  128006,
  882,
  128007,
  271,
  3923,
  374,
  279,
  5133,
  1990,
  1633,
  3428,
  73693,
  17,
  10,
  5990,
  11,
  393,
  3701,
  5990,
  11,
  323,
  14751,
  17,
  10,
  5990,
  30,
  128009,
  128006,
  78191,
  128007,
  271],
 'completion': 'Very low Mg2+ levels correspond to low PTH levels which in turn results in low Ca2+ levels.<|eot_id|>'}

In [ ]:
# Create a 95% Train / 5% Validation split (~1,700 rows for eval)
split_dataset = full_dataset.train_test_split(test_size=0.05, seed=3407)

train_data = split_dataset["train"]
eval_data = split_dataset["test"]

print(f"Training on {len(train_data)} rows. Evaluating on {len(eval_data)} rows.")

Training on 32257 rows. Evaluating on 1698 rows.


In [ ]:
# 5. Production Configuration
sft_config = SFTConfig(
    # training params
    output_dir="/content/drive/MyDrive/medical-llama3/llama-3.1-8b-instruct-lora-bf16",
    per_device_train_batch_size=64,
    gradient_accumulation_steps=1,  # Effective batch = 64
    num_train_epochs=1,
    learning_rate=1e-4,
    optim="adamw_torch",
    weight_decay=0.01,
    warmup_ratio=0.1,

    # LoRA params
    fp16=False,
    bf16=True,
    seed=3407,
    neftune_noise_alpha=5.0,
    dataset_text_field=None,
    completion_only_loss=True,
    max_seq_length=max_seq_length,
    dataset_num_proc=8,

    # eval params
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,
    report_to="wandb",
    run_name="batch-64_accum-1_lr-1e-4_decay-0.1",
)

# 6. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=sft_config,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=8):   0%|          | 0/32257 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=8):   0%|          | 0/1698 [00:00<?, ? examples/s]

Received unrecognized `WANDB_LOG_MODEL` setting value=true; so disabling `WANDB_LOG_MODEL`


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
# 7. Start Run & Monitor VRAM
# W&B automatically picks up GPU metrics, but if it lags,
# you can see it in the Colab "Resources" tab in real-time.
trainer.train()
wandb.finish()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,257 | Num Epochs = 1 | Total steps = 505
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 1 x 1) = 64
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.618868,0.601715


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/medical-llama3/llama-3.1-8b-instruct-lora-bf16/checkpoint-505/tokenizer_config.json.


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▆▇▇████
train/global_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▆▇▇████
train/grad_norm,▅▂▁▁▄▁▃▂▃▅▃▅▃▆▄▇▃▄▃█
train/learning_rate,▄███▇▇▆▆▅▅▅▄▄▃▃▃▂▂▁▁
train/loss,█▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
eval/loss,0.60171
eval/runtime,45.1097


In [ ]:
# 8. Merge to 16-bit model
print("💾 Saving LoRA adapters to Google Drive...")
save_directory = "/content/drive/MyDrive/medical-llama3/llama-3.1-lora-16bit"
model.save_pretrained_merged(save_directory, tokenizer, save_method="merged_16bit")
tokenizer.save_pretrained(save_directory)


print("✅ COMPLETE. Your weights are permanently secured.")

💾 Saving LoRA adapters to Google Drive...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/medical-llama3/production-16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/medical-llama3/production-16bit`:   0%|          | 0/4 [00:00<?, ?it/s]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/medical-llama3/production-16bit`:  25%|██▌       | 1/4 [00:12<00:37, 12.36s/it]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/medical-llama3/production-16bit`:  50%|█████     | 2/4 [00:29<00:30, 15.33s/it]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/medical-llama3/production-16bit`:  75%|███████▌  | 3/4 [00:44<00:15, 15.14s/it]
Unsloth: Copying 4 files from cache to `/content/drive/MyDrive/medical-llama3/production-16bit`: 100%|██████████| 4/4 [00:52<00:00, 13.25s/it]


Successfully copied all 4 files from cache to `/content/drive/MyDrive/medical-llama3/production-16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 26051.58it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [01:22<00:00, 20.55s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/medical-llama3/production-16bit`
✅ COMPLETE. Your weights are permanently secured.


# Compare original and fine-tuned LLM by removing and adding LORA adapters

In [ ]:
# 1. Define your test questions
test_prompts = [
    "What are the primary symptoms and first-line treatment for acute pericarditis?",
    "Explain the difference between Type 1 and Type 2 diabetes.",
    "What is the capital of France?" # Great control question to test general knowledge retention
]

# 2. The Robust Generator
def generate_response(question):
    # Match your EXACT training system prompt
    messages = [
        {"role": "system", "content": "You are a highly authoritative clinical diagnostic AI. Answer accurately and directly."},
        {"role": "user", "content": question}
    ]

    # Let the tokenizer handle the formatting, and trigger the "Green Light"
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generate the answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        use_cache=True,
        temperature=0.1, # Low temp is perfect for factual medical retrieval
        pad_token_id=tokenizer.eos_token_id
    )

    # The Mathematical Extraction (No string splitting required!)
    # We slice the output tensor to ignore all the prompt tokens, keeping only the newly generated tokens
    input_length = inputs['input_ids'].shape[1]
    generated_tokens = outputs[0][input_length:]

    # Decode only the AI's new words
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response.strip()

# 3. Execute the A/B Test
print("🧪 STARTING LORA A/B TEST...\n")
print("-" * 50)

for q in test_prompts:
    print(f"🗣️ QUESTION: {q}\n")

    # Test the base Llama 3.1 Instruct
    with model.disable_adapter():
        baseline_answer = generate_response(q)
        print(f"🤖 BASELINE LLAMA-3.1:\n{baseline_answer}\n")

    # Test your Medical Brain
    medical_answer = generate_response(q)
    print(f"🩺 YOUR MEDICAL MODEL:\n{medical_answer}\n")
    print("-" * 50)

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧪 STARTING LORA A/B TEST...

--------------------------------------------------
🗣️ QUESTION: What are the primary symptoms and first-line treatment for acute pericarditis?



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 BASELINE LLAMA-3.1:
**Primary Symptoms of Acute Pericarditis:**

1.  **Chest pain**: The most common symptom of acute pericarditis is chest pain, which is typically sharp, stabbing, or aching in nature. The pain is usually located in the center or left side of the chest and may radiate to the neck, shoulders, or arms.
2.  **Pleuritic chest pain**: The pain may worsen with deep breathing, coughing, or movement.
3.  **Fever**: Many patients with acute pericarditis experience fever, which can range from mild to high.
4.  **Fatigue**: Patients may feel weak and tired due to the inflammation and pain.
5.  **Short



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

🤖 BASELINE LLAMA-3.1:
Type 1 diabetes and Type 2 diabetes are two distinct forms of diabetes mellitus, a group of metabolic disorders characterized by high blood sugar levels. The primary difference between the two lies in their underlying causes, symptoms, and treatment approaches.

**Type 1 Diabetes:**

1.  **Autoimmune disease:** Type 1 diabetes is an autoimmune disease, where the body's immune system mistakenly attacks and destroys the insulin-producing beta cells in the pancreas. This results in a complete deficiency of insulin production.
2.  **Insulin dependence:** People with Type 1 diabetes require insulin therapy to control their blood sugar levels, as their bodies cannot produce enough insulin.
3.  **Symptoms:** Type 1 diabetes often develops rapidly, with



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🩺 YOUR MEDICAL MODEL:
Type 1 diabetes is an autoimmune disease in which the body's immune system attacks and destroys the cells in the pancreas that produce insulin, a hormone that regulates blood sugar levels. This results in a complete deficiency of insulin production, which can lead to high blood sugar levels and a range of complications if left untreated. Type 2 diabetes, on the other hand, is a metabolic disorder in which the body becomes resistant to insulin, making it difficult for glucose to enter the cells and leading to high blood sugar levels. While Type 2 diabetes is often associated with obesity and physical inactivity, it can also occur in people of normal weight and can be caused by a variety of factors, including genetics and certain medications.

--------------------------------------------------
🗣️ QUESTION: What is the capital of France?



Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 BASELINE LLAMA-3.1:
The capital of France is Paris.

🩺 YOUR MEDICAL MODEL:
The capital of France is Paris.

--------------------------------------------------
